# SOUL EXTER — LLM trading floor on a free Kaggle GPU

Six **open-source LLMs** sit in six glass cabins and vote on every trade the scanner finds.
No API keys, no paid endpoints — the weights download anonymously from Hugging Face.

**Setup**
1. Kaggle → *Create → Notebook* → **Settings → Accelerator → GPU T4 x2**, **Internet → On**.
2. Run the cells in order. Step 5 prints a public `https://…trycloudflare.com` URL — open it and you are on the floor.

**Model roster** (all ungated, 4-bit): Qwen2.5-7B · Mistral-7B-v0.3 · Zephyr-7B-β · Qwen2.5-3B ·
Phi-3.5-mini · **Qwen2.5-14B (CEO)**. First run downloads ~25 GB, so give it 10–15 minutes.
Kaggle gives you a 20 GB `/kaggle/working` quota — set `HF_HOME` there (cell 3) to keep weights across saves.

In [ ]:
# 1 ── confirm the machine
import subprocess, torch, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'no nvidia-smi')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| devices', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name}  {p.total_memory/1e9:.1f} GB')
print('cwd', os.getcwd(), '| disk free GB', round(__import__('shutil').disk_usage('/kaggle/working').free/1e9, 1))

In [ ]:
# 2 ── dependencies (a few minutes the first time)
!pip install -q --upgrade "transformers>=4.44" "accelerate>=0.33" bitsandbytes sentencepiece \
    fastapi "uvicorn[standard]" websockets httpx numpy ccxt pydantic
import transformers, bitsandbytes
print('transformers', transformers.__version__, '| bitsandbytes', bitsandbytes.__version__)

In [ ]:
# 3 ── get the code + cache weights on the writable volume
import os, sys, pathlib
REPO = 'https://github.com/Naserkhan07/soul_exter.git'
SRC  = '/kaggle/working/soul_exter'
os.environ['HF_HOME']        = '/kaggle/working/hf'      # ~25 GB of weights, survives a save
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
pathlib.Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

if not pathlib.Path(SRC).exists():
    !git clone --depth 1 {REPO} {SRC}
else:
    !cd {SRC} && git pull --ff-only
sys.path.insert(0, SRC)
print('code at', SRC)
print(sorted(p.name for p in pathlib.Path(SRC).iterdir()))

In [ ]:
# 4 ── smoke test one cabin before committing to the full floor
import os, sys
os.environ.update({
    'SOUL_MOCK_LLM': '0',            # use the real open-weight models
    'SOUL_MODEL_PROFILE': 'standard',
    'SOUL_LOAD_4BIT': '1',
    'SOUL_MARKET_SOURCE': 'auto',    # Binance public data, falls back to the simulator
    'SOUL_PORT': '8000',
})
sys.path.insert(0, '/kaggle/working/soul_exter')

from soul.config import load_config
from soul.brains.base import CABINS
from soul.brains import build_brains, model_for
cfg = load_config()
print('cabin roster:')
for c in CABINS:
    print(f'  {c.key:11s} <- {model_for(c, cfg.model_profile)}')

from soul.brains.local_hf import ModelPool, LocalHFBrain
pool = ModelPool(load_in_4bit=True, max_cached=6)
brain = LocalHFBrain(CABINS[0], pool, model_for(CABINS[0], cfg.model_profile))

import asyncio
from soul.models import TradeCandidate
t = TradeCandidate(symbol='SOL/USDT', side='LONG', strategy='MOMENTUM_BREAKOUT',
                   entry=152.4, stop=149.1, target=158.2, score=0.62,
                   features={'rsi': 58.0, 'vol_z': 2.4, 'atr_rank': 41.0, 'btc_ret_12': 0.9,
                             'rel_strength': 1.8, 'ema_stack': 1.0, 'regime': 1.0,
                             'corr_proxy': 1.55, 'atr_pct': 1.1, 'risk_pct': 2.17})   
ctx = {'market': {'price': 152.4, 'change_pct': 1.4, 'high': 154.0, 'low': 148.6,
                  'regime': 'risk-on', 'btc_change_pct': 1.1, 'vol_rank': 41.0, 'spread_bps': 4.2},
       'portfolio': {'equity': 15000, 'cash': 15000, 'open_pnl': 0, 'positions': [],
                     'closed': 0, 'win_rate': 0, 'realised_pnl': 0, 'gross_exposure': 0,
                     'max_positions': 8, 'planned_risk_pct': 0, 'risk_budget_pct': 0.75}}
v = asyncio.run(brain.judge(t, ctx, [], 1))
print('\nQUANT desk says:', v.verdict, f'{v.confidence:.0f}%', '|', v.reason[:220])
print('raw:', v.raw[:400])

In [ ]:
# 5 ── start the floor + public tunnel, then open the printed URL
import os, subprocess, threading, time, pathlib, socket

def free_port(p=8000):
    s = socket.socket()
    try:
        s.bind(('0.0.0.0', p)); return p
    except OSError:
        return 8001
    finally:
        s.close()

PORT = free_port()
os.environ['SOUL_PORT'] = str(PORT)

def run_server():
    os.system(f"cd /kaggle/working/soul_exter && SOUL_PORT={PORT} "
              f"python -m soul > /kaggle/working/floor.log 2>&1")
threading.Thread(target=run_server, daemon=True).start()
time.sleep(12)
print(open('/kaggle/working/floor.log').read()[-1500:])

# cloudflared: no account, no token, gives you an https URL that also carries websockets
CL = '/kaggle/working/cloudflared'
if not pathlib.Path(CL).exists():
    !wget -q -O {CL} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CL}

log = open('/kaggle/working/tunnel.log', 'w')
proc = subprocess.Popen([CL, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
                        stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    txt = pathlib.Path('/kaggle/working/tunnel.log').read_text()
    for tok in txt.replace('|', ' ').split():
        if tok.startswith('https://') and 'trycloudflare' in tok:
            url = tok; break
    if url: break
print('\n' + '=' * 70)
print('  OPEN THIS URL:  ', url)
print('=' * 70)
print('model downloads happen on the first trade — watch the cabins fill up.')

In [ ]:
# 6 ── keep-alive, monitoring, and GPU check while the floor runs
import time, subprocess, json, urllib.request
for i in range(60):
    time.sleep(30)
    try:
        s = json.load(urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/state', timeout=5))
        c, d = s['council'], s['desk']
        print(f"[{time.strftime('%H:%M:%S')}] reviews {c['reviews']} · entry {c['finalized']} · "
              f"exit {c['rejected']} · ceo {c['escalated']} · equity ${d['equity']:,.2f} "
              f"({d['return_pct']:+.2f}%) · open {d['open_positions']}", flush=True)
    except Exception as e:
        print('engine not answering yet:', e, flush=True)
    if i % 6 == 0:
        print(subprocess.run(['nvidia-smi', '--query-gpu=memory.used,utilization.gpu',
                              '--format=csv,noheader'], capture_output=True, text=True).stdout.strip(), flush=True)

### Tuning the pace

A 7B model on a T4 writes ~20 tokens/second, so a cabin verdict takes 10–25s. The engine runs the five
cabins in two waves and keeps several trades in flight, which lands around **1–3 completed trades per minute**.

Change these in cell 4 (or as `%env` before starting) and re-run cell 5:

| variable | effect |
|---|---|
| `SOUL_MODEL_PROFILE=low` | 3B/Phi-mini everywhere — ~3x faster, weaker reasoning |
| `SOUL_MODEL_CACHE=6` | how many 4-bit models stay resident (32 GB across 2×T4) |
| `SOUL_SCAN_SECONDS=45` | how often the scanner looks for new trades |
| `SOUL_MIN_SCORE=0.35` | stricter scanner threshold = fewer, cleaner trades |
| `SOUL_MAX_IN_FLIGHT=8` | trades allowed on the floor at once |
| `SOUL_WAVE_MODE=0` | strictly one cabin at a time (slower, but each cabin sees the previous one's verdict alone) |

If you want the trade packet to be reviewed by *exactly* the sequence QUANT → RISK → NEWS → MACRO →
COMPLIANCE → CEO, set `SOUL_WAVE_MODE=0`; that is the slowest and most literal reading of the flow.